# 04. Quantized Gradients: QSGD, TernGrad, and 1-bit Adam with Error Feedback

- **Category:** 14_quantisation / 05_kv_attention_optim_and_benchmarks
- **Lineage & Citations:** Alistarh et al. (2017, *QSGD: Communication-Efficient SGD via Gradient Quantization and Encoding*, NeurIPS); Wen et al. (2017, *TernGrad: Ternary Gradients to Reduce Communication in Distributed Deep Learning*, NeurIPS); Tang et al. (2021, *1-bit Adam: Communication Efficient Large-Scale Training with 1-bit Adam*, ICML).
- **Purpose:** Eliminate distributed network all-reduce communication bottlenecks by compressing gradients to 3-bit (QSGD), ternary $\{-1, 0, +1\}$ (TernGrad), and 1-bit momentum with Error Feedback (EF) residual compensation.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [03. Quantized Optimizer States 8bit and 4bit Adam](03_quantized_optimizer_states_8bit_and_4bit_adam.ipynb) $\to$ `🔗 Downstream Syntheses:` [05. HAWQ Mixed Precision Bit Allocation and Numeric Debugging](05_hawq_mixed_precision_bit_allocation_and_numeric_debugging.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT apply naive 1-bit quantization without an Error Feedback memory buffer (uncompensated 1-bit gradients accumulate permanent directional bias that destabilizes convergence); do NOT omit outlier clipping in ternary gradient quantization (gradient kurtosis causes extreme maximum scales that collapse interior coordinates to zero).

## Mathematical Foundations & Governing Equations

1. **QSGD Unbiased Norm Scaling & Rounding**:
   $$\xi_i = \frac{|g_i|}{\|g\|_2 + \epsilon}, \quad \tilde{\xi}_i = \frac{\lfloor s \cdot \xi_i \rceil}{s}, \quad Q_{\text{QSGD}}(g_i) = \text{sign}(g_i) \cdot \|g\|_2 \cdot \tilde{\xi}_i$$
2. **TernGrad Outlier-Clipped Ternary Quantization**:
   $$g_{\text{clip}} = \text{clamp}(g, -2.5\sigma, 2.5\sigma), \quad s_g = \max(|g_{\text{clip}}|), \quad b_i = \text{sign}(g_{\text{clip}, i}) \cdot s_g \cdot \mathbb{I}_{|g_{\text{clip}, i}|/s_g \ge 0.5}$$
3. **1-bit Adam with Error Feedback (EF)**:
   $$\tilde{m}_t = m_t + e_{t-1}, \quad s_t = \frac{1}{d}\sum_{j=1}^d |\tilde{m}_{t, j}|, \quad \hat{m}_t = s_t \cdot \text{sign}(\tilde{m}_t), \quad e_t = \tilde{m}_t - \hat{m}_t$$
4. **Error Feedback Telescoping Convergence**:
   $$\frac{1}{T}\sum_{t=1}^T \hat{m}_t = \frac{1}{T}\sum_{t=1}^T m_t - \frac{e_T}{T} \implies \lim_{T\to\infty} \left\|\frac{1}{T}\sum_{t=1}^T \hat{m}_t - \frac{1}{T}\sum_{t=1}^T m_t\right\|_2 = 0$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/14_quantisation/05_kv_attention_optim_and_benchmarks/04_quantized_gradients_qsgd_terngrad_and_1bit_adam_error_feedback.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} tensors from {WEIGHTS_PATH.name}")

In [ ]:
def qsgd_quantize(grad: torch.Tensor, num_levels: int = 7):
    norm = torch.norm(grad, p=2, dim=-1, keepdim=True)
    xi = torch.abs(grad) / (norm + 1e-8)
    tilde_xi = torch.round(num_levels * xi) / float(num_levels)
    return torch.sign(grad) * norm * tilde_xi, norm

def terngrad_quantize(grad: torch.Tensor, clip_factor: float = 2.5):
    mean = torch.mean(grad, dim=-1, keepdim=True)
    var = torch.mean((grad - mean) ** 2, dim=-1, keepdim=True)
    std = torch.sqrt(var + 1e-8)
    c = clip_factor * std
    g_clip = torch.clamp(grad, -c, c)
    scale = torch.amax(torch.abs(g_clip), dim=-1, keepdim=True)
    norm_g = torch.abs(g_clip) / (scale + 1e-8)
    return torch.sign(g_clip) * (norm_g >= 0.5).to(grad.dtype) * scale, scale

def onebit_adam_compress(momentum: torch.Tensor, error_buffer: torch.Tensor):
    m_tilde = momentum + error_buffer
    scale = torch.mean(torch.abs(m_tilde), dim=-1, keepdim=True)
    signs = torch.where(m_tilde >= 0, torch.ones_like(m_tilde), -torch.ones_like(m_tilde))
    m_hat = scale * signs
    return m_hat, m_tilde - m_hat, scale

def onebit_adam_step(param: torch.Tensor, m_hat: torch.Tensor, v: torch.Tensor, lr: float = 1e-3, eps: float = 1e-8, wd: float = 0.01):
    return param - lr * (m_hat / (torch.sqrt(v) + eps) + wd * param)

In [ ]:
# 1. Linear anchor parity
y_anchor = weights["test_input"] @ weights["weight"] + weights["bias"]
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

# 2. QSGD parity
q_grad, q_norm = qsgd_quantize(weights["grad"], num_levels=7)
assert float(torch.max(torch.abs(q_norm - weights["qsgd_norm"]))) < 1e-5
assert float(torch.max(torch.abs(q_grad - weights["qsgd_grad"]))) < 1e-5

# 3. TernGrad parity
t_grad, t_scale = terngrad_quantize(weights["grad"], clip_factor=2.5)
assert float(torch.max(torch.abs(t_scale - weights["terngrad_scale"]))) < 1e-5
assert float(torch.max(torch.abs(t_grad - weights["terngrad_grad"]))) < 1e-5

# 4. 1-bit Adam with Error Feedback step 1 and step 2 parity
m_hat_1, err_1, s_1 = onebit_adam_compress(weights["grad"], weights["error_0"])
assert float(torch.max(torch.abs(s_1 - weights["adam_scale_1"]))) < 1e-5
assert float(torch.max(torch.abs(m_hat_1 - weights["adam_m_hat_1"]))) < 1e-5
assert float(torch.max(torch.abs(err_1 - weights["adam_error_1"]))) < 1e-5

m_hat_2, err_2, s_2 = onebit_adam_compress(weights["grad_2"], err_1)
assert float(torch.max(torch.abs(s_2 - weights["adam_scale_2"]))) < 1e-5
assert float(torch.max(torch.abs(m_hat_2 - weights["adam_m_hat_2"]))) < 1e-5
assert float(torch.max(torch.abs(err_2 - weights["adam_error_2"]))) < 1e-5

# 5. Parameter update step parity
new_param = onebit_adam_step(weights["param"], m_hat_1, weights["v"], lr=1e-3, eps=1e-8, wd=0.01)
assert float(torch.max(torch.abs(new_param - weights["new_param"]))) < 1e-5
print("All QSGD, TernGrad, and 1-bit Adam assertions verified successfully against golden checkpoint!")

In [ ]:
# Error Feedback Telescoping Convergence: Accumulated SQNR Growth Across Steps
torch.manual_seed(42)
d_rows, d_cols = 1024, 1024
e_buf = torch.zeros(d_rows, d_cols)
true_sum = torch.zeros(d_rows, d_cols)
comp_sum = torch.zeros(d_rows, d_cols)

for step in range(1, 21):
    g_t = torch.randn(d_rows, d_cols) * 0.05
    true_sum += g_t
    m_t, e_buf, _ = onebit_adam_compress(g_t, e_buf)
    comp_sum += m_t
    if step in [1, 5, 10, 20]:
        sqnr = 20 * torch.log10(torch.norm(true_sum) / (torch.norm(true_sum - comp_sum) + 1e-12)).item()
        print(f"Step {step:2d} | Cumulative Momentum SQNR: {sqnr:.2f} dB (32.0x wire compression)")
        if step == 20:
            assert sqnr > 14.0, "Error feedback cumulative SQNR must exceed 14 dB by step 20!"

In [ ]:
# Latency Microbenchmark: Fused 1-bit Adam Momentum Compression (1024x1024 parameters)
n_rows, n_cols = 1024, 1024
g_bench = torch.randn(n_rows, n_cols) * 0.05
e_bench = torch.zeros(n_rows, n_cols)

t0 = time.perf_counter()
for _ in range(50):
    _, e_bench, _ = onebit_adam_compress(g_bench, e_bench)
t_comp = (time.perf_counter() - t0) / 50 * 1e3

print(f"Fused 1-bit Adam Compression ({n_rows*n_cols} parameters) Latency: {t_comp:.3f} ms")
assert t_comp < 20.0, "1-bit Adam compression must complete in <20 ms on CPU"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Telescoping Error Compensation**: Error Feedback stores residual quantization noise locally, driving the cumulative effective momentum drift to zero at an asymptotic rate of $\mathcal{O}(1/T)$.
2. **Outlier Clipping Tames Gradient Kurtosis**: Heavy-tailed gradient distributions destroy 3-level and low-bit representations unless clipped to $c \cdot \sigma$, preventing single-coordinate scale dominance.
3. **32x Distributed Wire Reduction**: 1-bit momentum compression slashes inter-node AllReduce bandwidth by $32\times$, unlocking linear scaling across clusters bound by interconnect latency.

---
### Synchronized Track Navigation
- [Companion Guide](../../../markdown/14_quantisation/05_kv_attention_optim_and_benchmarks/04_quantized_gradients_qsgd_terngrad_and_1bit_adam_error_feedback.md)
- [PyTorch Reference](../../../pytorch2/14_quantisation/05_kv_attention_optim_and_benchmarks/04_quantized_gradients_qsgd_terngrad_and_1bit_adam_error_feedback.py)
- [JAX Reference](../../../jax/14_quantisation/05_kv_attention_optim_and_benchmarks/04_quantized_gradients_qsgd_terngrad_and_1bit_adam_error_feedback.py)
- [Candle Reference](../../../candle/14_quantisation/05_kv_attention_optim_and_benchmarks/04_quantized_gradients_qsgd_terngrad_and_1bit_adam_error_feedback.rs)
- [Kernels Reference](../../../kernels/14_quantisation/05_kv_attention_optim_and_benchmarks/04_quantized_gradients_qsgd_terngrad_and_1bit_adam_error_feedback.rs)
- `🔗 Upstream Prerequisites:` [03. Quantized Optimizer States 8bit and 4bit Adam](03_quantized_optimizer_states_8bit_and_4bit_adam.ipynb)
- `🔗 Downstream Syntheses:` [05. HAWQ Mixed Precision Bit Allocation and Numeric Debugging](05_hawq_mixed_precision_bit_allocation_and_numeric_debugging.ipynb)